# Instalasi dan Import Library Python

Tujuan:
Perintah ini digunakan untuk menginstal library yang sering digunakan dalam analisis data time series, khususnya untuk forecasting data saham atau keuangan. Setiap library memiliki peran spesifik dalam proses berikut:

1. Mengambil data: yfinance.
2. Manipulasi data: pandas, numpy.
3. Modeling: pmdarima, keras, scikit-learn.
4. Visualisasi hasil: matplotlib.

In [ ]:
pip install yfinance pandas numpy pmdarima scikit-learn keras matplotlib

In [ ]:
pip install --upgrade pandas openpyxl

In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import matplotlib.pyplot as plt
import pmdarima as pm
from keras.models import Sequential
from keras.layers import LSTM, Dense
import tensorflow as tf
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.linear_model import LinearRegression
from sklearn.metrics import euclidean_distances
from openpyxl import Workbook
from openpyxl.drawing.image import Image
from openpyxl import load_workbook
import seaborn as sns
import shutil

# Step 1: Data Collection dan Data Understanding

Pada tahap ini dilakukan proses pengumpulan data dari website yahoo finance dan memahami karakteristik data berdasarkan deskripsi data dan grafik data aktual

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

# Name of Ticker Stock
ticker = 'BTPN.JK'

# Download Stock Price Data
data = yf.download(ticker, start="2021-02-01", end="2024-07-31", group_by='ticker')

# Rounding data to 4 decimal places
ticker_data = data.round(4)
print("Data Aktual BTPN.JK")
print(ticker_data)

# Displays data description per ticker with rounded numbers.
ticker_description = ticker_data.describe().round(4)
print("Data Deskripsi BTPN.JK")
print(ticker_description)

# Setting the visualization style
sns.set(style="whitegrid")

# Create a closing price plot for each stock
plt.figure(figsize=(14, 8))
plt.plot(data['Close'], label=ticker)

plt.title('Closing Prices of BTPN.JK')
plt.xlabel('Date')
plt.ylabel('Closing Price (IDR)')
plt.legend()

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file = os.path.join(output_dir2, 'BTPN_plot_data_aktual.png')
plt.savefig(plot_file)
plt.show()


# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the data and description to an Excel file
with pd.ExcelWriter(output_program, engine='openpyxl') as writer:
    ticker_data.to_excel(writer, sheet_name='Data Aktual', index=True)  # Set index=True to include date index
    ticker_description.to_excel(writer, sheet_name='Deskripsi Data', index=True)

# Explicitly print success message
print(f"File saved to {output_program}")


# Step 2: Data Preprocessing

Pada tahap ini dilakukan filtering/extract untuk fitur/variabel yang digunakan yaitu fitur harga saham "Close" sebagai dataset yang akan dimodelkan lalu dilakukan normalisasi pada dataset tersebut dan melakukan splitting data, yaitu membagi dataset menjadi data training dan data testing dengan perbandingan 80:20. Selanjutnya melakukan deteksi outlier dan cleaning data jika diperlukan

In [ ]:
#import os
#import pandas as pd
#import yfinance as yf
#from sklearn.preprocessing import MinMaxScaler

import yfinance as yf
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime

# Ambil data hingga akhir training
ticker = 'BTPN.JK'  # Gantilah dengan simbol saham yang sesuai
train_start = '2021-02-01'
train_end = '2023-11-07'

# Ambil data training
train_data_full = yf.download(ticker, start=train_start, end=train_end)

# Ambil data testing dari akhir training hingga hari ini
test_start = '2023-11-07'
test_end = datetime.today().strftime('%Y-%m-%d')  # Mengambil tanggal hari ini

test_data_full = yf.download(ticker, start=test_start, end=test_end)

# Gabungkan data training dan testing untuk normalisasi
data_full = pd.concat([train_data_full, test_data_full])

# Extract 'Close' prices and reshape for scaling
data_close = data_full['Close'].values.reshape(-1, 1)

# Normalisasi Data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data_close)

# Hitung jumlah data training agar tetap sama
train_size = len(train_data_full)
train_data, test_data = scaled_data[:train_size], scaled_data[train_size:]

# Cek hasil pemisahan data
print(f"Data Training: {train_data.shape[0]} data")
print(f"Data Testing: {test_data.shape[0]} data")

print(len(data_full.index))  # Should match len(data['Close'])
print(len(data_full['Close'].values))
print(len(scaled_data.flatten()))
print(len(['Train' if i < train_size else 'Test' for i in range(len(scaled_data))]))

In [ ]:
# Create DataFrame for displaying the results
df_result = pd.DataFrame({
    'Date': data_full.index,  # Original date from the data
    'Close': data_full['Close'].values,  # Original 'Close' values
    'Scaled_Close': scaled_data.flatten(),  # Normalized 'Close' values
    'Train_Flag': ['Train' if i < train_size else 'Test' for i in range(len(scaled_data))]  # Indicate Train/Test split
})

# Split the data into training and testing based on the flag
df_train = df_result[df_result['Train_Flag'] == 'Train'].reset_index(drop=True)
df_test = df_result[df_result['Train_Flag'] == 'Test'].reset_index(drop=True)

# Display the results
print("Data Splitting (Normalisasi):")
#print(df_result.head())  # Display the first few rows of the full result
print(df_result)

print("\nData Training:")
#print(df_train.head())  # Display the first few rows of the training data
print(df_train)

print("\nData Testing:")
#print(df_test.head())  # Display the first few rows of the testing data
print(df_test)

# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the DataFrame to Excel
with pd.ExcelWriter(output_program, engine='openpyxl', mode='a') as writer:
    df_result.to_excel(writer, sheet_name='Data_Splitting_Normalization', index=False)

# Explicitly print success message
print(f"File saved to {output_program}")


In [ ]:
# Deteksi Outlier menggunakan Interquartile Range (IQR)
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)  # Kuartil pertama
    Q3 = data[column].quantile(0.75)  # Kuartil ketiga
    IQR = Q3 - Q1  # Rentang antar kuartil
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Tandai outlier
    data['Outlier'] = (data[column] < lower_bound) | (data[column] > upper_bound)
    return data, lower_bound, upper_bound

# Terapkan fungsi deteksi outlier
df_result, lower_bound, upper_bound = detect_outliers_iqr(df_result, 'Close')

print(f"Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")
print("\nOutliers Detected:")
print(df_result[df_result['Outlier']])

with pd.ExcelWriter(output_program, engine='openpyxl', mode='a') as writer:
    df_result.to_excel(writer, sheet_name='Data_with_Outliers', index=False)


# Step 3: Data Modelling

Pada tahap ini dilakukan modelling untuk data training menggunakan 2 model dasar yaitu:

# 3-1. MODEL DASAR AUTO ARIMA

In [ ]:
import os
import pandas as pd
import pmdarima as pm
#from io import StringIO

# Step 3-1: Auto ARIMA Model
auto_arima_model = pm.auto_arima(train_data, seasonal=False, trace=True)

# Save Auto ARIMA Model Summary
model_summary = auto_arima_model.summary()

# Convert the summary to a string and then to a DataFrame
summary_text = model_summary.as_text()
summary_df = pd.DataFrame(summary_text.splitlines(), columns=["Auto ARIMA Model Summary"])

print("Auto ARIMA Model Summary BTPN.JK")
print(model_summary)

pred_arima_train = auto_arima_model.predict_in_sample()
pred_arima_test = auto_arima_model.predict(n_periods=len(test_data))
pred_arima_train = scaler.inverse_transform(pred_arima_train.reshape(-1, 1))
pred_arima_test = scaler.inverse_transform(pred_arima_test.reshape(-1, 1))


# 3-2. MODEL DASAR LSTM

In [ ]:
# Step 3-2: LSTM Model
def create_sequences(data, seq_length):
    sequences, labels = [], []
    for i in range(len(data) - seq_length):
        sequences.append(data[i:i + seq_length])
        labels.append(data[i + seq_length])
    return np.array(sequences), np.array(labels)

seq_length = 1  # Example sequence length
train_sequences, train_labels = create_sequences(train_data, seq_length)
test_sequences, test_labels = create_sequences(test_data, seq_length)

lstm_model = Sequential()
lstm_model.add(LSTM(units=50, return_sequences=True, input_shape=(seq_length, 1)))
lstm_model.add(LSTM(units=50))
lstm_model.add(Dense(1))
lstm_model.compile(optimizer='adam', loss='mse')

# Train LSTM model and capture the history
history = lstm_model.fit(train_sequences, train_labels, epochs=100, batch_size=32, verbose=1)

# Print LSTM model summary
print("\nLSTM Model Summary:")
lstm_model.summary()

# Generate LSTM predictions
pred_lstm_train = lstm_model.predict(train_sequences)
pred_lstm_test = lstm_model.predict(test_sequences)
pred_lstm_train = scaler.inverse_transform(pred_lstm_train)
pred_lstm_test = scaler.inverse_transform(pred_lstm_test)

# Plot loss over epochs
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss', color='blue')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

Setelah diperoleh model ARIMA dan model LSTM, hasil output dari masing-masing model akan digabungkan menjadi model hybrid ARIMA LSTM menggunakan pendekatan smoothing regression Ordinary Least Square (OLS) Regression, Lowess Linear Regression, Lowess Quadratic Regression dan Kernel Triangular Regression

# 3-3. MODEL HYBRID ARIMA-LSTM

# 3-3-1. Model Hybrid ARIMA-LSTM with OLS Linear Regression

In [ ]:
# Step 3-3-1: Combining Predictions with Linear Regression OLS (Hybrid ARIMA-LSTM with OLS)
combined_length_train = min(len(pred_arima_train), len(pred_lstm_train))
combined_length_test = min(len(pred_arima_test), len(pred_lstm_test))

# Truncate predictions to ensure same length for stacking
pred_arima_train = pred_arima_train[:combined_length_train]
pred_lstm_train = pred_lstm_train[:combined_length_train]
pred_arima_test = pred_arima_test[:combined_length_test]
pred_lstm_test = pred_lstm_test[:combined_length_test]

X_combined_train = np.column_stack((pred_arima_train, pred_lstm_train))
X_combined_test = np.column_stack((pred_arima_test, pred_lstm_test))

y_actual_train = scaler.inverse_transform(train_data[seq_length:seq_length+combined_length_train])
y_actual_test = scaler.inverse_transform(test_data[:combined_length_test])

# Adding a constant to the regression model (intercept)
X_combined_train_sm = sm.add_constant(X_combined_train)
X_combined_test_sm = sm.add_constant(X_combined_test, has_constant='add')

# Handle the case when the number of features in X_combined_test_sm is less than in X_combined_train_sm
if X_combined_test_sm.shape[1] < X_combined_train_sm.shape[1]:
    # Add a zero column to X_combined_test_sm
    zeros_column = np.zeros((X_combined_test_sm.shape[0], 1))
    X_combined_test_sm = np.hstack([X_combined_test_sm, zeros_column])

# Linear Regression model using statsmodels
reg_model = sm.OLS(y_actual_train, X_combined_train_sm).fit()

# Print Linear Regression model summary using statsmodels
print("\nLinear Regression Model Summary:")
print(reg_model.summary())

# Generate final predictions
final_predictions_train = reg_model.predict(X_combined_train_sm)
final_predictions_test = reg_model.predict(X_combined_test_sm)

# 3-3-2. Model Hybrid ARIMA-LSTM with Lowess Linear Regression

In [ ]:
# Step 3-3-2: Combining Predictions with Local Linear Regression (Hybrid ARIMA-LSTM with Lowess Linear Regression)
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import euclidean_distances

def Lowess_2d(X1, X2, Y, x1_new, x2_new, frac=0.1):
    """
    Lowess Linear Regression dengan dua variabel bebas.

    Args:
        X1, X2: Variabel bebas (numpy arrays)
        Y: Variabel dependen (numpy array)
        x1_new, x2_new: Titik baru untuk prediksi
        frac: Fraksi data untuk digunakan dalam smoothing (bandwidth)

    Returns:
        Prediksi untuk titik baru (y_pred_new)
    """
    # Gabungkan X1 dan X2 menjadi satu matriks
    X = np.vstack((X1, X2)).T

    # Hitung jarak Euclidean dari titik prediksi ke semua titik dalam dataset
    distances = euclidean_distances(X, np.array([[x1_new, x2_new]]))
    print("Distances")
    print(distances)

    # Tentukan bandwidth sebagai persentil ke-frac dari jarak
    k = int(frac * len(X1))
    bandwidth = np.sort(distances, axis=0)[k]
    print("Nilai k")
    print(k)
    print("Bandwidth")
    print(bandwidth)

    # Hitung bobot menggunakan kernel tricube
    weights = (1 - (distances / bandwidth)**3)**3
    weights[distances > bandwidth] = 0  # Atur bobot di luar bandwidth menjadi nol

    # Tampilkan matriks diagonal bobot
    W = np.diag(weights.flatten())
    print("Matriks diagonal bobot (W):")
    print(W)

    # Tampilkan ukuran matriks bobot
    print(f"Ukuran matriks bobot: {W.shape}")

    # Lakukan regresi linear dengan bobot
    reg = LinearRegression()
    reg.fit(X, Y, sample_weight=weights.flatten())

    # Tampilkan parameter regresi
    beta_0 = reg.intercept_
    beta_1, beta_2 = reg.coef_
    print(f"Parameter regresi: beta_0 = {beta_0:.4f}, beta_1 = {beta_1:.4f}, beta_2 = {beta_2:.4f}")

    # Bentuk persamaan regresi linear berbobot
    print(f"Bentuk persamaan regresi: y = {beta_0:.4f} + {beta_1:.4f} * x1 + {beta_2:.4f} * x2")


    # Prediksi untuk titik baru
    y_pred_new = reg.predict(np.array([[x1_new, x2_new]]))

    return y_pred_new

# Fungsi untuk melakukan prediksi pada semua titik di dalam dataset
def Lowess_predict_all(X1, X2, Y, frac=0.1):
    """
    Lakukan prediksi Lowess Regression untuk semua titik dalam dataset.

    Args:
        X1, X2: Variabel bebas (numpy arrays)
        Y: Variabel dependen (numpy array)
        frac: Fraksi data untuk digunakan dalam smoothing (bandwidth)

    Returns:
        Prediksi untuk semua titik (y_pred_all)
    """
    y_pred_all = []

    for x1_new, x2_new in zip(X1, X2):
        y_pred_new = Lowess_2d(X1, X2, Y, x1_new, x2_new, frac=frac)
        y_pred_all.append(y_pred_new)

    return np.array(y_pred_all).flatten()

# Fungsi untuk summary dari model
def summary_model(X1_train, X2_train, Y_train, X1_test, X2_test, Y_test, frac=0.1):
    # Prediksi untuk data train
    y_pred_train = Lowess_predict_all(X1_train, X2_train, Y_train, frac)

    # Prediksi untuk data test
    y_pred_test = Lowess_predict_all(X1_test, X2_test, Y_test, frac)

    # Menghitung metrik evaluasi
    mse_train = mean_squared_error(Y_train, y_pred_train)
    mae_train = mean_absolute_error(Y_train, y_pred_train)
    r2_train = r2_score(Y_train, y_pred_train)

    mse_test = mean_squared_error(Y_test, y_pred_test)
    mae_test = mean_absolute_error(Y_test, y_pred_test)
    r2_test = r2_score(Y_test, y_pred_test)

    print("Summary of Lowess 2D Model:")

    print(f"Train MSE: {mse_train:.4f}")
    print(f"Train MAE: {mae_train:.4f}")
    print(f"Train R^2: {r2_train:.4f}")
    print(f"Test MSE: {mse_test:.4f}")
    print(f"Test MAE: {mae_test:.4f}")
    print(f"Test R^2: {r2_test:.4f}")

    return y_pred_train, y_pred_test

# Contoh penggunaan
X1_train = np.array(pred_arima_train).flatten()  # Ganti dengan output ARIMA
X2_train = np.array(pred_lstm_train).flatten()   # Ganti dengan output LSTM
Y_train = np.array(y_actual_train).flatten()     # Ganti dengan nilai aktual
y_pred_all_Lowess_train = Lowess_predict_all(X1_train, X2_train, Y_train)

X1_test = np.array(pred_arima_test).flatten()  # Ganti dengan output ARIMA
X2_test = np.array(pred_lstm_test).flatten()   # Ganti dengan output LSTM
Y_test = np.array(y_actual_test).flatten()     # Ganti dengan nilai aktual
y_pred_all_Lowess_test = Lowess_predict_all(X1_test, X2_test, Y_test)

# Mendapatkan summary dan prediksi
# y_pred_all_Lowess_train, y_pred_all_Lowess_test = summary_model(X1_train, X2_train, Y_train, X1_test, X2_test, Y_test, frac=0.1)


# 3-3-3. Model Hybrid ARIMA-LSTM with Lowess Quadratic Regression

In [ ]:
# Step 3-3-3: Combining Predictions with Local Polynomial Regression (Hybrid ARIMA-LSTM with Lowess Quadratic Regression)
# Apply quadratic polynomial features
poly = PolynomialFeatures(degree=2)
X_combined_train_poly = poly.fit_transform(X_combined_train)
X_combined_test_poly = poly.fit_transform(X_combined_test)

# Perform local regression for each point
def local_polynomial_regression(X, y, frac):
    n = len(X)
    y_smoothed = np.zeros(n)
    for i in range(n):
        distances = np.abs(X[:, 0] - X[i, 0])
        weights = np.exp(-distances / (2*frac*frac))
        weighted_regressor = LinearRegression()
        weighted_regressor.fit(X, y, sample_weight=weights)
        y_smoothed[i] = weighted_regressor.predict(X[i].reshape(1, -1))
    return y_smoothed

frac = 0.1 # Fraction of data used for local fitting
lowess_train_poly = local_polynomial_regression(X_combined_train_poly, y_actual_train.flatten(), frac)
lowess_test_poly = local_polynomial_regression(X_combined_test_poly, y_actual_test.flatten(), frac)

# 3-3-4. Model Hybrid ARIMA-LSTM with Kernel Triangular Regression

In [ ]:
# Step 3-3-4: Combining Predictions with Kernel Triangular Regression (Hybrid ARIMA-LSTM with Kernel Triangular Regression)
# Fungsi Kernel Triangular Regression 2D
def kernel_triangular_2d(X1, X2, Y, x1_new, x2_new, frac=0.1):
    # Menggabungkan X1 dan X2 menjadi matriks 2D
    X = np.vstack((X1, X2)).T

    # Hitung jarak Euclidean antara titik-titik data dan titik baru
    distances = euclidean_distances(X, np.array([[x1_new, x2_new]]))

    # Tentukan bandwidth berdasarkan persentase terdekat (frac)
    k = int(frac * len(X1))
    bandwidth = np.sort(distances, axis=0)[k]  # Bandwidth sebagai jarak terdekat ke-k

    # Hitung bobot menggunakan kernel segitiga
    weights = np.maximum(1 - (distances / bandwidth), 0)

    # Fit linear regression menggunakan bobot ini
    reg = LinearRegression()
    reg.fit(X, Y, sample_weight=weights.flatten())

    # Prediksi untuk titik baru
    y_pred_new = reg.predict(np.array([[x1_new, x2_new]]))

    return y_pred_new

# Fungsi untuk melakukan prediksi pada semua titik di dalam dataset
def kernel_triangular_predict_all(X1, X2, Y, frac=0.1):
    y_pred_all = []
    for x1_new, x2_new in zip(X1, X2):
        y_pred_new = kernel_triangular_2d(X1, X2, Y, x1_new, x2_new, frac=frac)
        y_pred_all.append(y_pred_new)
    return np.array(y_pred_all).flatten()

# Fungsi untuk summary dari model
def summary_model_kernel(X1_train, X2_train, Y_train, X1_test, X2_test, Y_test, frac=0.1):
    # Prediksi untuk data train
    y_pred_train = kernel_triangular_predict_all(X1_train, X2_train, Y_train, frac)

    # Prediksi untuk data test
    y_pred_test = kernel_triangular_predict_all(X1_test, X2_test, Y_test, frac)

    # Menghitung metrik evaluasi untuk data training
    mse_train = mean_squared_error(Y_train, y_pred_train)
    mae_train = mean_absolute_error(Y_train, y_pred_train)
    r2_train = r2_score(Y_train, y_pred_train)

    # Menghitung metrik evaluasi untuk data testing
    mse_test = mean_squared_error(Y_test, y_pred_test)
    mae_test = mean_absolute_error(Y_test, y_pred_test)
    r2_test = r2_score(Y_test, y_pred_test)

    # Menampilkan hasil
    print("Summary of Kernel Triangular 2D Model:")
    print(f"Train MSE: {mse_train:.4f}")
    print(f"Train MAE: {mae_train:.4f}")
    print(f"Train R^2: {r2_train:.4f}")
    print(f"Test MSE: {mse_test:.4f}")
    print(f"Test MAE: {mae_test:.4f}")
    print(f"Test R^2: {r2_test:.4f}")

    return y_pred_train, y_pred_test

y_pred_train, y_pred_test = summary_model_kernel(X1_train, X2_train, Y_train, X1_test, X2_test, Y_test, frac=0.1)

# Step 4: Membuat Hasil Prediksi Tiap Model

Pada tahap ini dihitung hasil prediksi/peramalan menggunakan tiap model

In [ ]:
# Create DataFrame for displaying the results
dates_train = data_full.index[seq_length:seq_length+combined_length_train]
dates_test = data_full.index[len(train_data):len(train_data)+combined_length_test]

df_train = pd.DataFrame({
    'Date': dates_train,
    'Actual': y_actual_train.flatten(),
    'ARIMA_Prediction': pred_arima_train[:combined_length_train].flatten(),
    'LSTM_Prediction': pred_lstm_train[:combined_length_train].flatten(),
    'Hybrid OLS_Prediction' : final_predictions_train.flatten(),
    'Hybrid Lowess_Prediction': y_pred_all_Lowess_train.flatten(),
    'Hybrid Lowess^2_Prediction': lowess_train_poly.flatten(),
    'Hybrid Kernel Prediction': y_pred_train.flatten()

})

df_test = pd.DataFrame({
    'Date': dates_test,
    'Actual': y_actual_test.flatten(),
    'ARIMA_Prediction': pred_arima_test[:combined_length_test].flatten(),
    'LSTM_Prediction': pred_lstm_test[:combined_length_test].flatten(),
    'Hybrid OLS_Prediction' : final_predictions_test.flatten(),
    'Hybrid Lowess_Prediction': y_pred_all_Lowess_test.flatten(),
    'Hybrid Lowess^2 Prediction': lowess_test_poly.flatten(),
    'Hybrid Kernel Prediction': y_pred_test.flatten()

})

print("Training Data Predictions:")
print(df_train)
print("\nTesting Data Predictions:")
print(df_test)

# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

if df_train.empty or df_test.empty:
    print("Error: DataFrame is empty. No data to save.")
else:
    with pd.ExcelWriter(output_program, engine='openpyxl', mode='a') as writer:
        df_train.to_excel(writer, sheet_name='Train_Predictions', index=False)
        df_test.to_excel(writer, sheet_name='Test_Predictions', index=False)

    print(f"File saved to {output_program}")



# Step 5: Evaluasi Model

Pada tahap ini dilakukan evaluasi terhadap hasil prediksi tiap model untuk data training dan data testing, yaitu:

# 5.1. Menghitung Error Prediksi Tiap Model

In [ ]:
# Hasil Error Prediksi (menghitung error absolut)
df_error_train = pd.DataFrame({
    'Date': dates_train,
    'ARIMA_Error': abs(df_train['Actual'] - df_train['ARIMA_Prediction']),
    'LSTM_Error': abs(df_train['Actual'] - df_train['LSTM_Prediction']),
    'Hybrid OLS_Error': abs(df_train['Actual'] - df_train['Hybrid OLS_Prediction']),
    'Hybrid Lowess_Error': abs(df_train['Actual'] - df_train['Hybrid Lowess_Prediction']),
    'Hybrid Lowess^2_Error': abs(df_train['Actual'] - df_train['Hybrid Lowess^2_Prediction']),
    'Hybrid Kernel_Error': abs(df_train['Actual'] - df_train['Hybrid Kernel Prediction'])
})

df_error_test = pd.DataFrame({
    'Date': dates_test,
    'ARIMA_Error': abs(df_test['Actual'] - df_test['ARIMA_Prediction']),
    'LSTM_Error': abs(df_test['Actual'] - df_test['LSTM_Prediction']),
    'Hybrid OLS_Error': abs(df_test['Actual'] - df_test['Hybrid OLS_Prediction']),
    'Hybrid Lowess_Error': abs(df_test['Actual'] - df_test['Hybrid Lowess_Prediction']),
    'Hybrid Lowess^2_Error': abs(df_test['Actual'] - df_test['Hybrid Lowess^2 Prediction']),
    'Hybrid Kernel_Error': abs(df_test['Actual'] - df_test['Hybrid Kernel Prediction'])
})

print("Training Data Error Predictions:")
print(df_error_train)
print("\nTesting Data Error Predictions:")
print(df_error_test)

# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Simpan tabel ke Excel
with pd.ExcelWriter(output_program, engine='openpyxl', mode='a') as writer:
    df_error_train.to_excel(writer, sheet_name='Train_Error_Predictions', index=False)
    df_error_test.to_excel(writer, sheet_name='Test_Error_Predictions', index=False)

# Explicitly print success message
print(f"File saved to {output_program}")

# 5.2. Menghitung Metriks Evaluasi Tiap Model

Pada tahap ini dilakukan perhitungan metriks evaluasi berupa nilai Mean Square Error (MSE), Root Mean Square Error (RMSE), Mean Absolute Error (MAE) dan Mean Absolute Percentage Error (MAPE) untuk tiap model

In [ ]:
import pywt
from openpyxl import Workbook
from openpyxl.drawing.image import Image
from openpyxl import load_workbook
import shutil

def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return mse, rmse, mae, mape

# Evaluate ARIMA model
mse_arima_train, rmse_arima_train, mae_arima_train, mape_arima_train = evaluate_model(y_actual_train, pred_arima_train[:combined_length_train])
mse_arima_test, rmse_arima_test, mae_arima_test, mape_arima_test = evaluate_model(y_actual_test, pred_arima_test[:combined_length_test])

# Evaluate LSTM model
mse_lstm_train, rmse_lstm_train, mae_lstm_train, mape_lstm_train = evaluate_model(y_actual_train, pred_lstm_train[:combined_length_train])
mse_lstm_test, rmse_lstm_test, mae_lstm_test, mape_lstm_test = evaluate_model(y_actual_test, pred_lstm_test[:combined_length_test])

# Evaluate Hybrid ARIMA-LSTM OLS model
mse_ols_train, rmse_ols_train, mae_ols_train, mape_ols_train = evaluate_model(y_actual_train, final_predictions_train)
mse_ols_test, rmse_ols_test, mae_ols_test, mape_ols_test = evaluate_model(y_actual_test, final_predictions_test)

# Evaluate Hybrid ARIMA-LSTM Lowess Linear Regression model
mse_Lowess_train, rmse_Lowess_train, mae_Lowess_train, mape_Lowess_train = evaluate_model(Y_train, y_pred_all_Lowess_train)
mse_Lowess_test, rmse_Lowess_test, mae_Lowess_test, mape_Lowess_test = evaluate_model(Y_test, y_pred_all_Lowess_test)

# Evaluate Hybrid ARIMA-LSTM Lowess^2 Regression model
mse_lowess_train_poly, rmse_lowess_train_poly, mae_lowess_train_poly, mape_lowess_train_poly = evaluate_model(y_actual_train, lowess_train_poly)
mse_lowess_test_poly, rmse_lowess_test_poly, mae_lowess_test_poly, mape_lowess_test_poly = evaluate_model(y_actual_test, lowess_test_poly)

# Evaluate Hybrid ARIMA-LSTM Kernel model
mse_kernel_train, rmse_kernel_train, mae_kernel_train, mape_kernel_train = evaluate_model(Y_train, y_pred_train)
mse_kernel_test, rmse_kernel_test, mae_kernel_test, mape_kernel_test = evaluate_model(Y_test, y_pred_test)

# Create DataFrame for metrics
metrics_data = {
    'Model': ['ARIMA', 'ARIMA', 'LSTM', 'LSTM', 'Hybrid OLS', 'Hybrid OLS', 'Hybrid Lowess', 'Hybrid Lowess', 'Hybrid Lowess^2', 'Hybrid Lowess^2','Hybrid Kernel','Hybrid Kernel'],
    'Data': ['Train', 'Test', 'Train', 'Test', 'Train', 'Test', 'Train', 'Test', 'Train', 'Test','Train','Test'],
    'MSE': [mse_arima_train, mse_arima_test, mse_lstm_train, mse_lstm_test, mse_ols_train, mse_ols_test, mse_Lowess_train, mse_Lowess_test, mse_lowess_train_poly, mse_lowess_test_poly, mse_kernel_train, mse_kernel_test],
    'RMSE': [rmse_arima_train, rmse_arima_test, rmse_lstm_train, rmse_lstm_test, rmse_ols_train, rmse_ols_test, rmse_Lowess_train, rmse_Lowess_test, rmse_lowess_train_poly, rmse_lowess_test_poly, rmse_kernel_train, rmse_kernel_test],
    'MAE': [mae_arima_train, mae_arima_test, mae_lstm_train, mae_lstm_test, mae_ols_train, mae_ols_test, mae_Lowess_train, mae_Lowess_test, mae_lowess_train_poly, mae_lowess_test_poly, mae_kernel_train, mae_kernel_test],
    'MAPE': [mape_arima_train, mape_arima_test, mape_lstm_train, mape_lstm_test, mape_ols_train, mape_ols_test, mape_Lowess_train, mape_Lowess_test, mape_lowess_train_poly, mape_lowess_test_poly, mape_kernel_train, mape_kernel_test]
}

df_metrics = pd.DataFrame(metrics_data)
print("\nEvaluation Metrics:")
print(df_metrics)

# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the metrics to the same Excel file
with pd.ExcelWriter(output_program, engine='openpyxl', mode='a') as writer:
    df_metrics.to_excel(writer, sheet_name='Evaluation_Metrics', index=False)

# Explicitly print success message
print(f"File saved to {output_program}")

# Step 6: Visualisasi Hasil Prediksi Tiap Model

Pada tahap ini dilakukan visualisasi yaitu menggambarkan grafik nilai aktual dan grafik hasil prediksi tiap model untuk data training dan data testing

# 6.1. Grafik Data Training

In [ ]:
# Plot Data Train Model Hybrid ARIMA-LSTM OLS
plt.figure(figsize=(14, 7))
# Plot actual values
plt.plot(data_full.index[:train_size], scaler.inverse_transform(train_data), label='Actual Train', color='blue')
# Plot ARIMA model predictions
plt.plot(dates_train, pred_arima_train[:combined_length_train], label='ARIMA Train Predictions', color='green')
# Plot LSTM model predictions
plt.plot(dates_train, pred_lstm_train[:combined_length_train], label='LSTM Train Predictions', color='yellow')
# Plot OLS model predictions
plt.plot(dates_train, final_predictions_train, label='Hybrid ARIMA-LSTM OLS Train Predictions', color='red')

plt.legend()
plt.title("Stock Price Predictions Data Train")
plt.xlabel("Date")
plt.ylabel("Stock Price")

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file1 = os.path.join(output_dir2, 'stock_predictions_BTPN_plot_data_train_Hybrid_OLS.png')
plt.savefig(plot_file1)
plt.show()

In [ ]:
# Plot Data Train Model Hybrid ARIMA-LSTM Lowess Linier
plt.figure(figsize=(14, 7))
# Plot actual values
plt.plot(data_full.index[:train_size], scaler.inverse_transform(train_data), label='Actual Train', color='blue')
# Plot ARIMA model predictions
plt.plot(dates_train, pred_arima_train[:combined_length_train], label='ARIMA Train Predictions', color='orange')
# Plot LSTM model predictions
plt.plot(dates_train, pred_lstm_train[:combined_length_train], label='LSTM Train Predictions', color='green')
# Plot Lowess model predictions
plt.plot(dates_train, y_pred_all_Lowess_train, label='Hybrid ARIMA-LSTM Lowess Train Predictions', color='red')

plt.legend()
plt.title("Stock Price Predictions Data Train")
plt.xlabel("Date")
plt.ylabel("Stock Price")

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file2 = os.path.join(output_dir2, 'stock_predictions_BTPN_plot_data_train_Hybrid_Lowess_Linier.png')
plt.savefig(plot_file2)
plt.show()

In [ ]:
# Plot Data Train Model Hybrid ARIMA-LSTM Lowess Kuadrat
plt.figure(figsize=(14, 7))
# Plot actual values
plt.plot(data_full.index[:train_size], scaler.inverse_transform(train_data), label='Actual Train', color='blue')
# Plot ARIMA model predictions
plt.plot(dates_train, pred_arima_train[:combined_length_train], label='ARIMA Train Predictions', color='green')
# Plot LSTM model predictions
plt.plot(dates_train, pred_lstm_train[:combined_length_train], label='LSTM Train Predictions', color='yellow')
# Plot Lowess2 model predictions
plt.plot(dates_train, lowess_train_poly, label='Hybrid ARIMA-LSTM Lowess^2 Train Predictions', color='red')

plt.legend()
plt.title("Stock Price Predictions Data Train")
plt.xlabel("Date")
plt.ylabel("Stock Price")

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file3 = os.path.join(output_dir2, 'stock_predictions_BTPN_plot_data_train_Hybrid_Lowess_Quadrat.png')
plt.savefig(plot_file3)
plt.show()

In [ ]:
# Plot Data Train Model Hybrid ARIMA-LSTM Kernel
plt.figure(figsize=(14, 7))
# Plot actual values
plt.plot(data_full.index[:train_size], scaler.inverse_transform(train_data), label='Actual Train', color='blue')
# Plot ARIMA model predictions
plt.plot(dates_train, pred_arima_train[:combined_length_train], label='ARIMA Train Predictions', color='green')
# Plot LSTM model predictions
plt.plot(dates_train, pred_lstm_train[:combined_length_train], label='LSTM Train Predictions', color='yellow')
# Plot Kernel model predictions
plt.plot(dates_train, y_pred_train, label='Hybrid ARIMA-LSTM Kernel Train Predictions', color='red')

plt.legend()
plt.title("Stock Price Predictions Data Train")
plt.xlabel("Date")
plt.ylabel("Stock Price")

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file4 = os.path.join(output_dir2, 'stock_predictions_BTPN_plot_data_train_Hybrid_Kernel.png')
plt.savefig(plot_file4)
plt.show()

# 6.2. Grafik Data Testing

In [ ]:
# Plot Data Test Model Hybrid ARIMA-LSTM OLS
plt.figure(figsize=(14, 7))
# Plot actual values
plt.plot(data_full.index[train_size:], scaler.inverse_transform(test_data), label='Actual Test', color='blue')
# Plot ARIMA model predictions
plt.plot(dates_test, pred_arima_test[:combined_length_test], label='ARIMA Test Predictions', color='green')
# Plot LSTM model predictions
plt.plot(dates_test, pred_lstm_test[:combined_length_test], label='LSTM Test Predictions', color='yellow')
# Plot OLS model predictions
plt.plot(dates_test, final_predictions_test[:combined_length_test], label='Hybrid ARIMA-LSTM OLS Test Predictions', color='red')

plt.legend()
plt.title("Stock Price Predictions Data Test")
plt.xlabel("Date")
plt.ylabel("Stock Price")

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file5 = os.path.join(output_dir2, 'stock_predictions_BTPN_plot_data_test_Hybrid_OLS.png')
plt.savefig(plot_file5)
plt.show()

In [ ]:
# Plot Data Test Model Hybrid ARIMA-LSTM Lowess Linier
plt.figure(figsize=(14, 7))
# Plot actual values
plt.plot(data_full.index[train_size:], scaler.inverse_transform(test_data), label='Actual Test', color='blue')
# Plot ARIMA model predictions
plt.plot(dates_test, pred_arima_test[:combined_length_test], label='ARIMA Test Predictions', color='green')
# Plot LSTM model predictions
plt.plot(dates_test, pred_lstm_test[:combined_length_test], label='LSTM Test Predictions', color='yellow')
# Plot Lowess model predictions
plt.plot(dates_test, y_pred_all_Lowess_test[:combined_length_test], label='Hybrid ARIMA-LSTM Lowess Test Predictions', color='red')

plt.legend()
plt.title("Stock Price Predictions Data Test")
plt.xlabel("Date")
plt.ylabel("Stock Price")

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file6 = os.path.join(output_dir2, 'stock_predictions_BTPN_plot_data_test_Hybrid_Lowess_Linier.png')
plt.savefig(plot_file6)
plt.show()

In [ ]:
# Plot Data Test Model Hybrid ARIMA-LSTM Lowess Kuadrat
plt.figure(figsize=(14, 7))
# Plot actual values
plt.plot(data_full.index[train_size:], scaler.inverse_transform(test_data), label='Actual Test', color='blue')
# Plot ARIMA model predictions
plt.plot(dates_test, pred_arima_test[:combined_length_test], label='ARIMA Test Predictions', color='green')
# Plot LSTM model predictions
plt.plot(dates_test, pred_lstm_test[:combined_length_test], label='LSTM Test Predictions', color='yellow')
# Plot Lowess2 model predictions
plt.plot(dates_test, lowess_test_poly[:combined_length_test], label='Hybrid ARIMA-LSTM Lowess^2 Test Predictions', color='red')

plt.legend()
plt.title("Stock Price Predictions Data Test")
plt.xlabel("Date")
plt.ylabel("Stock Price")

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file7 = os.path.join(output_dir2, 'stock_predictions_BTPN_plot_data_test_Hybrid_Lowess_Quadrat.png')
plt.savefig(plot_file7)
plt.show()

In [ ]:
# Plot Data Train Model Hybrid ARIMA-LSTM Kernel
plt.figure(figsize=(14, 7))
# Plot actual values
plt.plot(data_full.index[train_size:], scaler.inverse_transform(test_data), label='Actual Test', color='blue')
# Plot ARIMA model predictions
plt.plot(dates_test, pred_arima_test[:combined_length_test], label='ARIMA Test Predictions', color='green')
# Plot LSTM model predictions
plt.plot(dates_test, pred_lstm_test[:combined_length_test], label='LSTM Test Predictions', color='yellow')
# Plot Kernel model predictions
plt.plot(dates_test, y_pred_test, label='Hybrid ARIMA-LSTM Kernel Test Predictions', color='red')

plt.legend()
plt.title("Stock Price Predictions Data Test")
plt.xlabel("Date")
plt.ylabel("Stock Price")

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file8 = os.path.join(output_dir2, 'stock_predictions_BTPN_plot_data_test_Hybrid_Kernel.png')
plt.savefig(plot_file8)
plt.show()

# 6.3. Grafik Error Prediksi Data Training dan Data Testing

In [ ]:
# Hitung error mutlak (absolute error) untuk setiap model pada data training
df_train['Error_ARIMA'] = abs(df_train['Actual'] - df_train['ARIMA_Prediction'])
df_train['Error_LSTM'] = abs(df_train['Actual'] - df_train['LSTM_Prediction'])
df_train['Error_Hybrid_OLS'] = abs(df_train['Actual'] - df_train['Hybrid OLS_Prediction'])
df_train['Error_Hybrid_Lowess'] = abs(df_train['Actual'] - df_train['Hybrid Lowess_Prediction'])
df_train['Error_Hybrid_Lowess^2'] = abs(df_train['Actual'] - df_train['Hybrid Lowess^2_Prediction'])
df_train['Error_Hybrid_Kernel'] = abs(df_train['Actual'] - df_train['Hybrid Kernel Prediction'])

# Hitung error mutlak (absolute error) untuk setiap model pada data testing
df_test['Error_ARIMA'] = abs(df_test['Actual'] - df_test['ARIMA_Prediction'])
df_test['Error_LSTM'] = abs(df_test['Actual'] - df_test['LSTM_Prediction'])
df_test['Error_Hybrid_OLS'] = abs(df_test['Actual'] - df_test['Hybrid OLS_Prediction'])
df_test['Error_Hybrid_Lowess'] = abs(df_test['Actual'] - df_test['Hybrid Lowess_Prediction'])
df_test['Error_Hybrid_Lowess^2'] = abs(df_test['Actual'] - df_test['Hybrid Lowess^2 Prediction'])
df_test['Error_Hybrid_Kernel'] = abs(df_test['Actual'] - df_test['Hybrid Kernel Prediction'])

# Membuat DataFrame baru untuk Error Prediksi
error_columns = ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_OLS', 'Error_Hybrid_Lowess', 'Error_Hybrid_Lowess^2', 'Error_Hybrid_Kernel']
df_train_error = df_train[['Date'] + error_columns]
df_test_error = df_test[['Date'] + error_columns]

# Visualisasi grafik dari error prediksi
plt.figure(figsize=(14, 7))

# Plot Error untuk masing-masing model pada data train
for col in error_columns:
    plt.plot(df_train['Date'], df_train[col], label=f'Train {col}', linewidth=1)

plt.title('Train Error Predictions for Each Model')
plt.xlabel('Date')
plt.ylabel('Absolute Error')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

# Ensure the directory exists
output_dir2 = 'Output Program/Grafik'
os.makedirs(output_dir2, exist_ok=True)
# output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Save the plot as a PNG file
plot_file9 = os.path.join(output_dir2, 'BTPN_plot_error_data_train_Hybrid.png')
plt.savefig(plot_file9)
plt.show()

plt.figure(figsize=(14, 7))

# Plot Error untuk masing-masing model pada data test
for col in error_columns:
    plt.plot(df_test['Date'], df_test[col], label=f'Test {col}', linewidth=1)

plt.title('Test Error Predictions for Each Model')
plt.xlabel('Date')
plt.ylabel('Absolute Error')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()

# Save the plot as a PNG file
plot_file10 = os.path.join(output_dir2, 'BTPN_plot_error_data_test_Hybrid.png')
plt.savefig(plot_file10)
plt.show()


print(f"Saved error results and plots to {output_program}")

In [ ]:
import os
import matplotlib.pyplot as plt

# Kolom untuk setiap kombinasi grafik
error_combinations = [
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_OLS'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Lowess'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Lowess^2'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Kernel']
]

titles = [
    'Error Predictions: ARIMA, LSTM, Hybrid OLS',
    'Error Predictions: ARIMA, LSTM, Hybrid Lowess Linear',
    'Error Predictions: ARIMA, LSTM, Hybrid Lowess Quadratic',
    'Error Predictions: ARIMA, LSTM, Hybrid Kernel'
]

# Fungsi untuk membuat plot terpisah untuk setiap kombinasi model
def plot_individual_errors(df_train_error, df_test_error, error_combinations, titles, output_dir):
    for i, errors in enumerate(error_combinations):
        # Membuat figure baru dengan ukuran besar
        plt.figure(figsize=(10, 6), dpi=120)

        # Plot Error pada data training
        for error in errors:
            plt.plot(df_train_error['Date'], df_train_error[error], label=f'Train {error}', linewidth=1)

        # Plot Error pada data testing
        for error in errors:
            plt.plot(df_test_error['Date'], df_test_error[error], label=f'Test {error}', linestyle='--', linewidth=1)

        # Set judul, label, dan elemen grafis lainnya
        plt.title(titles[i])
        plt.xlabel('Date')
        plt.ylabel('Absolute Error')
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True)

        # Mengatur layout agar rapi
        plt.tight_layout()

        # Simpan setiap grafik sebagai gambar terpisah
        plot_file = os.path.join(output_dir, f'BTPN_plot_error_{titles[i].replace(" ", "_").replace(":", "")}.png')
        plt.savefig(plot_file)
        plt.show()

# Pastikan direktori output ada
output_dir = 'Output Program/Grafik'
os.makedirs(output_dir, exist_ok=True)

# Panggil fungsi untuk memvisualisasikan dan menyimpan setiap grafik sebagai file terpisah
plot_individual_errors(df_train_error, df_test_error, error_combinations, titles, output_dir)


In [ ]:
import os
import matplotlib.pyplot as plt

# Fungsi untuk mengatur skala sumbu Y berdasarkan error maksimum dan minimum
def set_ylimit(errors):
    # Menemukan nilai minimum dan maksimum dari error untuk penyesuaian skala Y
    min_error = min([errors[col].min() for col in errors.columns if col != 'Date'])
    max_error = max([errors[col].max() for col in errors.columns if col != 'Date'])
    return min_error, max_error

# Fungsi untuk membuat plot terpisah untuk data training dan testing
def plot_individual_errors(df_train_error, df_test_error, error_combinations, titles, output_dir):
    for i, errors in enumerate(error_combinations):

        # Plot untuk data training
        plt.figure(figsize=(10, 6), dpi=120)

        # Plot Error pada data training
        for error in errors:
            plt.plot(df_train_error['Date'], df_train_error[error], label=f'Train {error}', linewidth=1)

        # Atur skala sumbu Y untuk data training
        min_error, max_error = set_ylimit(df_train_error[errors])
        plt.ylim(min_error, max_error)

        # Set judul, label, dan elemen grafis lainnya
        plt.title(f'Train {titles[i]}')
        plt.xlabel('Date')
        plt.ylabel('Absolute Error')
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True)

        # Mengatur layout agar rapi
        plt.tight_layout()

        # Simpan grafik data training sebagai gambar
        plot_file_train = os.path.join(output_dir, f'BTPN_plot_train_error_{titles[i].replace(" ", "_").replace(":", "")}.png')
        plt.savefig(plot_file_train)
        plt.show()

        # Plot untuk data testing
        plt.figure(figsize=(10, 6), dpi=120)

        # Plot Error pada data testing
        for error in errors:
            plt.plot(df_test_error['Date'], df_test_error[error], label=f'Test {error}', linestyle='--', linewidth=1)

        # Atur skala sumbu Y untuk data testing
        min_error, max_error = set_ylimit(df_test_error[errors])
        plt.ylim(min_error, max_error)

        # Set judul, label, dan elemen grafis lainnya
        plt.title(f'Test {titles[i]}')
        plt.xlabel('Date')
        plt.ylabel('Absolute Error')
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True)

        # Mengatur layout agar rapi
        plt.tight_layout()

        # Simpan grafik data testing sebagai gambar
        plot_file_test = os.path.join(output_dir, f'BTPN_plot_test_error_{titles[i].replace(" ", "_").replace(":", "")}.png')
        plt.savefig(plot_file_test)
        plt.show()

# Daftar kombinasi error yang akan diplot untuk setiap grup
error_combinations = [
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_OLS'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Lowess'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Lowess^2'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Kernel']
]

# Judul untuk setiap grafik
titles = [
    'ARIMA, LSTM, Hybrid OLS',
    'ARIMA, LSTM, Hybrid Lowess Linear',
    'ARIMA, LSTM, Hybrid Lowess Quadratic',
    'ARIMA, LSTM, Hybrid Kernel'
]

# Pastikan direktori output ada
output_dir = 'Output Program/Grafik'
os.makedirs(output_dir, exist_ok=True)

# Panggil fungsi untuk memvisualisasikan dan menyimpan grafik
plot_individual_errors(df_train_error, df_test_error, error_combinations, titles, output_dir)


In [ ]:
import numpy as np

# Fungsi yang disesuaikan untuk menambahkan buffer pada skala sumbu Y
def set_ylimit_with_buffer(errors, buffer=0.05):
    # Menemukan nilai minimum dan maksimum dari error untuk penyesuaian skala Y
    min_error = min([errors[col].min() for col in errors.columns if col != 'Date'])
    max_error = max([errors[col].max() for col in errors.columns if col != 'Date'])

    # Tambahkan buffer pada skala, buffer adalah persentase dari rentang (default 5%)
    error_range = max_error - min_error
    min_error -= buffer * error_range
    max_error += buffer * error_range

    return min_error, max_error

# Fungsi untuk membuat plot terpisah untuk data training dan testing dengan buffer skala
def plot_individual_errors(df_train_error, df_test_error, error_combinations, titles, output_dir):
    for i, errors in enumerate(error_combinations):

        # Plot untuk data training
        plt.figure(figsize=(10, 6), dpi=120)

        # Plot Error pada data training
        for error in errors:
            plt.plot(df_train_error['Date'], df_train_error[error], label=f'Train {error}', linewidth=1)

        # Atur skala sumbu Y untuk data training dengan buffer
        min_error, max_error = set_ylimit_with_buffer(df_train_error[errors])
        plt.ylim(min_error, max_error)

        # Set judul, label, dan elemen grafis lainnya
        plt.title(f'Train {titles[i]}')
        plt.xlabel('Date')
        plt.ylabel('Absolute Error')
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True)

        # Mengatur layout agar rapi
        plt.tight_layout()

        # Simpan grafik data training sebagai gambar
        plot_file_train = os.path.join(output_dir, f'BTPN_plot_train_error_{titles[i].replace(" ", "_").replace(":", "")}.png')
        plt.savefig(plot_file_train)
        plt.show()

        # Plot untuk data testing
        plt.figure(figsize=(10, 6), dpi=120)

        # Plot Error pada data testing
        for error in errors:
            plt.plot(df_test_error['Date'], df_test_error[error], label=f'Test {error}', linestyle='--', linewidth=1)

        # Atur skala sumbu Y untuk data testing dengan buffer
        min_error, max_error = set_ylimit_with_buffer(df_test_error[errors])
        plt.ylim(min_error, max_error)

        # Set judul, label, dan elemen grafis lainnya
        plt.title(f'Test {titles[i]}')
        plt.xlabel('Date')
        plt.ylabel('Absolute Error')
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True)

        # Mengatur layout agar rapi
        plt.tight_layout()

        # Simpan grafik data testing sebagai gambar
        plot_file_test = os.path.join(output_dir, f'BTPN_plot_test_error_{titles[i].replace(" ", "_").replace(":", "")}.png')
        plt.savefig(plot_file_test)
        plt.show()

# Daftar kombinasi error yang akan diplot untuk setiap grup
error_combinations = [
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_OLS'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Lowess'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Lowess^2'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Kernel']
]

# Judul untuk setiap grafik
titles = [
    'ARIMA, LSTM, Hybrid OLS',
    'ARIMA, LSTM, Hybrid Lowess Linear',
    'ARIMA, LSTM, Hybrid Lowess Quadratic',
    'ARIMA, LSTM, Hybrid Kernel'
]

# Pastikan direktori output ada
output_dir = 'Output Program/Grafik'
os.makedirs(output_dir, exist_ok=True)

# Panggil fungsi untuk memvisualisasikan dan menyimpan grafik
plot_individual_errors(df_train_error, df_test_error, error_combinations, titles, output_dir)


In [ ]:
import numpy as np

# Fungsi yang disesuaikan untuk menambahkan buffer pada skala sumbu Y
def set_ylimit_with_buffer(errors, buffer=0.05):
    # Menemukan nilai minimum dan maksimum dari error untuk penyesuaian skala Y
    min_error = min([errors[col].min() for col in errors.columns if col != 'Date'])
    max_error = max([errors[col].max() for col in errors.columns if col != 'Date'])

    # Tambahkan buffer pada skala, buffer adalah persentase dari rentang (default 5%)
    error_range = max_error - min_error
    min_error -= buffer * error_range
    max_error += buffer * error_range

    return min_error, max_error

# Fungsi untuk memplot error berdasarkan interval 100 periode
def plot_errors_by_interval(df_error, error_columns, interval=100, data_type='Train'):
    # Tentukan berapa banyak grafik yang perlu dibuat berdasarkan panjang data
    total_periods = len(df_error)
    num_plots = (total_periods // interval) + (1 if total_periods % interval != 0 else 0)

    # Loop untuk memotong data setiap 100 periode dan memplot grafik
    for i in range(num_plots):
        start_period = i * interval
        end_period = min((i + 1) * interval, total_periods)  # Pastikan periode tidak melebihi total data

        # Ambil subset data untuk periode ini
        df_subset = df_error.iloc[start_period:end_period]

        # Plot untuk setiap kombinasi error model
        for j, errors in enumerate(error_columns):
            plt.figure(figsize=(10, 6), dpi=120)

            # Plot error untuk setiap model
            for error in errors:
                plt.plot(df_subset['Date'], df_subset[error], label=f'{data_type} {error}', linewidth=1)

            # Atur skala Y dengan buffer
            min_error, max_error = set_ylimit_with_buffer(df_subset[errors])
            plt.ylim(min_error, max_error)

            # Atur judul, label, dan elemen grafis lainnya
            plt.title(f'{data_type} Errors {errors} for Period {start_period + 1}-{end_period}')
            plt.xlabel('Date')
            plt.ylabel('Absolute Error')
            plt.legend()
            plt.xticks(rotation=45)
            plt.grid(True)

            # Mengatur layout agar rapi
            plt.tight_layout()

            # Simpan grafik sebagai gambar
            output_file = os.path.join(output_dir, f'{data_type}_Error_Plot_Period_{start_period + 1}_{end_period}_Model_{j + 1}.png')
            plt.savefig(output_file)
            plt.show()

# Daftar kombinasi error yang akan diplot untuk setiap grup
error_combinations = [
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_OLS'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Lowess'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Lowess^2'],
    ['Error_ARIMA', 'Error_LSTM', 'Error_Hybrid_Kernel']
]

# Pastikan direktori output ada
output_dir = 'Output Program/Grafik'
os.makedirs(output_dir, exist_ok=True)

# Plot grafik data training dan testing, setiap 100 periode
plot_errors_by_interval(df_train_error, error_combinations, interval=100, data_type='Train')
plot_errors_by_interval(df_test_error, error_combinations, interval=100, data_type='Test')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sample every 50 periods
train_indices = np.arange(0, len(df_error_train), 50)
test_indices = np.arange(0, len(df_error_test), 50)

# Set up bar width
bar_width = 0.1

# Create positions for the bars on the X axis
x_train = np.arange(len(train_indices))
x_test = np.arange(len(test_indices))

# Create figure and subplots
fig, axs = plt.subplots(2, 1, figsize=(20, 14), dpi=120)

# Plot for Training Data
axs[0].bar(x_train - 2.5 * bar_width, df_error_train['ARIMA_Error'].iloc[train_indices], bar_width, label='ARIMA Train', color='blue')
axs[0].bar(x_train - 1.5 * bar_width, df_error_train['LSTM_Error'].iloc[train_indices], bar_width, label='LSTM Train', color='orange')
axs[0].bar(x_train - 0.5 * bar_width, df_error_train['Hybrid OLS_Error'].iloc[train_indices], bar_width, label='Hybrid OLS Train', color='green')
axs[0].bar(x_train + 0.5 * bar_width, df_error_train['Hybrid Lowess_Error'].iloc[train_indices], bar_width, label='Hybrid Lowess Train', color='red')
axs[0].bar(x_train + 1.5 * bar_width, df_error_train['Hybrid Lowess^2_Error'].iloc[train_indices], bar_width, label='Hybrid Lowess^2 Train', color='purple')
axs[0].bar(x_train + 2.5 * bar_width, df_error_train['Hybrid Kernel_Error'].iloc[train_indices], bar_width, label='Hybrid Kernel Train', color='brown')

# Set training plot titles and labels
axs[0].set_title('Training Data Errors every 50 Time Periods', fontsize=16)
axs[0].set_xlabel('Time Period (Train)', fontsize=12)
axs[0].set_ylabel('Error', fontsize=12)
axs[0].set_xticks(x_train)
axs[0].set_xticklabels(df_error_train['Date'].iloc[train_indices], rotation=45, ha='right')
axs[0].grid(True)
axs[0].legend(fontsize=12)

# Plot for Testing Data
axs[1].bar(x_test - 2.5 * bar_width, df_error_test['ARIMA_Error'].iloc[test_indices], bar_width, label='ARIMA Test', color='blue')
axs[1].bar(x_test - 1.5 * bar_width, df_error_test['LSTM_Error'].iloc[test_indices], bar_width, label='LSTM Test', color='orange')
axs[1].bar(x_test - 0.5 * bar_width, df_error_test['Hybrid OLS_Error'].iloc[test_indices], bar_width, label='Hybrid OLS Test', color='green')
axs[1].bar(x_test + 0.5 * bar_width, df_error_test['Hybrid Lowess_Error'].iloc[test_indices], bar_width, label='Hybrid Lowess Test', color='red')
axs[1].bar(x_test + 1.5 * bar_width, df_error_test['Hybrid Lowess^2_Error'].iloc[test_indices], bar_width, label='Hybrid Lowess^2 Test', color='purple')
axs[1].bar(x_test + 2.5 * bar_width, df_error_test['Hybrid Kernel_Error'].iloc[test_indices], bar_width, label='Hybrid Kernel Test', color='brown')

# Set testing plot titles and labels
axs[1].set_title('Testing Data Errors every 50 Time Periods', fontsize=16)
axs[1].set_xlabel('Time Period (Test)', fontsize=12)
axs[1].set_ylabel('Error', fontsize=12)
axs[1].set_xticks(x_test)
axs[1].set_xticklabels(df_error_test['Date'].iloc[test_indices], rotation=45, ha='right')
axs[1].grid(True)
axs[1].legend(fontsize=12)

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Take the last 20 periods from the training data
train_indices_last_20 = np.arange(len(df_error_train) - 20, len(df_error_train))

# Take the last 20 periods from the testing data
test_indices_last_20 = np.arange(len(df_error_test) - 20, len(df_error_test))

# Set up bar width
bar_width = 0.1

# Create positions for the bars on the X axis
x_train = np.arange(len(train_indices_last_20))
x_test = np.arange(len(test_indices_last_20))

# Create figure and subplots
fig, axs = plt.subplots(2, 1, figsize=(20, 14), dpi=120)

# Plot for Training Data (Last 20 periods)
axs[0].bar(x_train - 2.5 * bar_width, df_error_train['ARIMA_Error'].iloc[train_indices_last_20], bar_width, label='ARIMA Train', color='blue')
axs[0].bar(x_train - 1.5 * bar_width, df_error_train['LSTM_Error'].iloc[train_indices_last_20], bar_width, label='LSTM Train', color='orange')
axs[0].bar(x_train - 0.5 * bar_width, df_error_train['Hybrid OLS_Error'].iloc[train_indices_last_20], bar_width, label='Hybrid OLS Train', color='green')
axs[0].bar(x_train + 0.5 * bar_width, df_error_train['Hybrid Lowess_Error'].iloc[train_indices_last_20], bar_width, label='Hybrid Lowess Train', color='red')
axs[0].bar(x_train + 1.5 * bar_width, df_error_train['Hybrid Lowess^2_Error'].iloc[train_indices_last_20], bar_width, label='Hybrid Lowess^2 Train', color='purple')
axs[0].bar(x_train + 2.5 * bar_width, df_error_train['Hybrid Kernel_Error'].iloc[train_indices_last_20], bar_width, label='Hybrid Kernel Train', color='brown')

# Set training plot titles and labels
axs[0].set_title('Training Data Errors (Last 20 Periods)', fontsize=16)
axs[0].set_xlabel('Time Period (Train)', fontsize=12)
axs[0].set_ylabel('Error', fontsize=12)
axs[0].set_xticks(x_train)
axs[0].set_xticklabels(df_error_train['Date'].iloc[train_indices_last_20], rotation=45, ha='right')
axs[0].grid(True)
axs[0].legend(fontsize=12)

# Plot for Training Data (Last 20 periods)
axs[1].bar(x_test - 2.5 * bar_width, df_error_train['ARIMA_Error'].iloc[test_indices_last_20], bar_width, label='ARIMA Train', color='blue')
axs[1].bar(x_test - 1.5 * bar_width, df_error_train['LSTM_Error'].iloc[test_indices_last_20], bar_width, label='LSTM Train', color='orange')
axs[1].bar(x_test - 0.5 * bar_width, df_error_train['Hybrid OLS_Error'].iloc[test_indices_last_20], bar_width, label='Hybrid OLS Train', color='green')
axs[1].bar(x_test + 0.5 * bar_width, df_error_train['Hybrid Lowess_Error'].iloc[test_indices_last_20], bar_width, label='Hybrid Lowess Train', color='red')
axs[1].bar(x_test + 1.5 * bar_width, df_error_train['Hybrid Lowess^2_Error'].iloc[test_indices_last_20], bar_width, label='Hybrid Lowess^2 Train', color='purple')
axs[1].bar(x_test + 2.5 * bar_width, df_error_train['Hybrid Kernel_Error'].iloc[test_indices_last_20], bar_width, label='Hybrid Kernel Train', color='brown')

# Set testing plot titles and labels
axs[1].set_title('Testing Data Errors (Last 20 Periods)', fontsize=16)
axs[1].set_xlabel('Time Period (Test)', fontsize=12)
axs[1].set_ylabel('Error', fontsize=12)
axs[1].set_xticks(x_test)
axs[1].set_xticklabels(df_error_test['Date'].iloc[test_indices_last_20], rotation=45, ha='right')
axs[1].grid(True)
axs[1].legend(fontsize=12)

# Adjust layout to prevent overlap
plt.tight_layout()

# Show the plot
plt.show()


# 6.4. Gambar Metriks Evaluasi

In [ ]:
# Visualization of Metrics
def plot_metrics(metrics_data, metric_name, ylabel):
    models = ['ARIMA', 'LSTM', 'Hybrid OLS', 'Hybrid Lowess', 'Hybrid Lowess^2', 'Hybrid Kernel']
    train_metrics = [metrics_data.loc[metrics_data['Model'] == model][metric_name].values[0] for model in models]
    test_metrics = [metrics_data.loc[metrics_data['Model'] == model][metric_name].values[1] for model in models]

    x = np.arange(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot bars for train and test metrics
    rects1 = ax.bar(x - width/2, train_metrics, width, label='Train')
    rects2 = ax.bar(x + width/2, test_metrics, width, label='Test')

    # Plot lines connecting the bars for train and test metrics
    ax.plot(x - width/2, train_metrics, color='blue', marker='o', label='Train Line')
    ax.plot(x + width/2, test_metrics, color='orange', marker='o', label='Test Line')

    # Set labels, title, and ticks
    ax.set_ylabel(ylabel)
    ax.set_title(f'BTPN {metric_name} by Model and Data Split')
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend()

    # Add labels on top of the bars
    ax.bar_label(rects1, padding=3)
    ax.bar_label(rects2, padding=3)

    fig.tight_layout()

# Save the plot as a PNG file
    metric_plot_file = os.path.join(output_dir2, f'BTPN {metric_name}_plot.png')
    plt.savefig(metric_plot_file)
    plt.show()

    return metric_plot_file


In [ ]:
metric_plot_files = []
metric_plot_files.append(plot_metrics(df_metrics, 'MSE', 'Mean Squared Error'))

In [ ]:
metric_plot_files.append(plot_metrics(df_metrics, 'RMSE', 'Root Mean Squared Error'))

In [ ]:
metric_plot_files.append(plot_metrics(df_metrics, 'MAE', 'Mean Absolute Error'))

In [ ]:
metric_plot_files.append(plot_metrics(df_metrics, 'MAPE', 'Mean Absolute Percentage Error'))

In [ ]:
# Example usage for MSE
plot_metrics(df_metrics, 'MSE', 'Mean Squared Error')

# Example usage for RMSE
plot_metrics(df_metrics, 'RMSE', 'Root Mean Squared Error')

# Example usage for MAE
plot_metrics(df_metrics, 'MAE', 'Mean Absolute Error')

# Example usage for MAPE
plot_metrics(df_metrics, 'MAPE', 'Mean Absolute Percentage Error')


# Step 7: Dokumentasi Hasil

# 7.1. Menyimpan Grafik Aktual ke Excel

In [ ]:
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# Define output directory and file
# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Load the existing workbook
wb = load_workbook(output_program)

# Create a new sheet for plots
ws = wb.create_sheet('Plots Aktual')

# Add the main plot Data AKtual
img = Image(plot_file)
img.anchor = 'A1'
ws.add_image(img)

# Menyisipkan gambar pertama ke dalam Excel
img = Image(plot_file)
ws.add_image(img, 'A1')  # Memasukkan gambar ke sel A1

# Menyimpan file Excel
wb.save(output_program)


# 7.2. Menyimpan Grafik Prediksi ke Excel

In [ ]:
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# Define output directory and file
# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Load the existing workbook
wb = load_workbook(output_program)

# Create a new sheet for plots
ws = wb.create_sheet('Plots Prediksi')

# List of plot file paths and their positions
plot_files = [
    (plot_file1, 'A1'),
    (plot_file2, 'A40'),
    (plot_file3, 'A80'),
    (plot_file4, 'A120'),
    (plot_file5, 'A160'),
    (plot_file6, 'A200'),
    (plot_file7, 'A240'),
    (plot_file8, 'A280')
]

# Add images to the new sheet
for plot_file, position in plot_files:
    img = Image(plot_file)
    img.anchor = position
    ws.add_image(img)

# Save the modified Excel file
wb.save(output_program)


# 7.3. Menyimpan Grafik Prediksi Data Training ke Excel

In [ ]:
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# Define output directory and file
# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Load the existing workbook
wb = load_workbook(output_program)

# Create a new sheet for Data Training plots
ws = wb.create_sheet('Plots Data Training')

# List of plot file paths and their positions
plot_files = [
    (plot_file1, 'A1'),
    (plot_file2, 'A40'),
    (plot_file3, 'A80'),
    (plot_file4, 'A120')
]

# Add images to the new sheet
for plot_file, position in plot_files:
    img = Image(plot_file)
    img.anchor = position
    ws.add_image(img)

# Save the modified Excel file
wb.save(output_program)


# 7.4. Menyimpan Grafik Prediksi Data Testing ke Excel

In [ ]:
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# Define output directory and file
# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Load the existing workbook
wb = load_workbook(output_program)

# Create a new sheet for Data Testing plots
ws = wb.create_sheet('Plots Data Testing')

# List of plot file paths and their positions
plot_files = [
    (plot_file5, 'A1'),
    (plot_file6, 'A40'),
    (plot_file7, 'A80'),
    (plot_file8, 'A120')
]

# Add images to the new sheet
for plot_file, position in plot_files:
    img = Image(plot_file)
    img.anchor = position
    ws.add_image(img)

# Save the modified Excel file
wb.save(output_program)


# 7.5. Menyimpan Grafik Error Prediksi Data Training dan Data Testing ke Excel

In [ ]:
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# Define output directory and file
# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')

# Load the existing workbook
wb = load_workbook(output_program)

# Create a new sheet for Error Prediction plots
ws = wb.create_sheet('Plots Error Prediksi')

# List of plot file paths and their positions
plot_files = [
    (plot_file9, 'A1'),
    (plot_file10, 'A40')
]

# Add images to the new sheet
for plot_file, position in plot_files:
    img = Image(plot_file)
    img.anchor = position
    ws.add_image(img)

# Save the modified Excel file
wb.save(output_program)


In [ ]:
# Add metric plots
# Ensure the directory exists
output_dir1 = 'Output Program/Tabel'
os.makedirs(output_dir1, exist_ok=True)
output_program = os.path.join(output_dir1, 'Program_Forecast_Data_BTPN_final.xlsx')
wb = load_workbook(output_program)
ws = wb.create_sheet('Plots Metric Evaluation')

row = 5
for metric_plot_file in metric_plot_files:
    img = Image(metric_plot_file)
    img.anchor = f'A{row}'
    ws.add_image(img)
    row += 30

# Menyimpan file Excel
wb.save(output_program)

print(f"Saved plots to {output_program}")

# For downloading the file
#shutil(output_program, 'Program_Forecast_Data_BTPN_final.xlsx')

print("File 'Program_Forecast_Data_BTPN_final.xlsx' is ready for download.")